In [13]:
%pip install python-pptx

StatementMeta(, dbab9ab1-6a0f-4194-8b8b-8ed1717414c7, 19, Finished, Available, Finished, True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 8.6 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



## Import and scan directories

In [6]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
from datetime import datetime
from pptx import Presentation
import json


# Root path
BASE_PATH = "/lakehouse/default/Files/SharepointGeneral/Customers"

# Output table
OUTPUT_TABLE = "proposal_file_inventory"
SLIDES_OUTPUT_TABLE = "proposal_slide_content_raw"

# File types we care about
VALID_EXTENSIONS = [".pptx", ".docx", ".pdf"]

StatementMeta(, dbab9ab1-6a0f-4194-8b8b-8ed1717414c7, 8, Finished, Available, Finished, False)

## Function for iterative scanning

In [7]:
def scan_directory(path):
    records = []
    
    try:
        with os.scandir(path) as entries:
            for entry in entries:
                try:
                    stat = entry.stat()
                    
                    records.append({
                        "path": entry.path,
                        "file_name": entry.name,
                        "extension": os.path.splitext(entry.name)[1].lower(),
                        "size_bytes": stat.st_size,
                        "modified_time": datetime.fromtimestamp(stat.st_mtime),
                        "is_directory": entry.is_dir()
                    })
                    
                    # Recurse
                    if entry.is_dir():
                        records.extend(scan_directory(entry.path))
                        
                except Exception as e:
                    records.append({
                        "path": entry.path,
                        "file_name": entry.name,
                        "extension": None,
                        "size_bytes": None,
                        "modified_time": None,
                        "is_directory": None,
                        "error": str(e)
                    })
                    
    except Exception as e:
        print(f"Error scanning {path}: {e}")
    
    return records

StatementMeta(, dbab9ab1-6a0f-4194-8b8b-8ed1717414c7, 9, Finished, Available, Finished, False)

## Execute iterative scan and load as dataframe

In [8]:
raw_records = scan_directory(BASE_PATH)

df = spark.createDataFrame(raw_records)

StatementMeta(, dbab9ab1-6a0f-4194-8b8b-8ed1717414c7, 10, Finished, Available, Finished, False)

In [9]:
df = (
    df
    .withColumn("scan_timestamp", current_timestamp())
    
    # Extract customer name (assuming folder structure: /Customers/{customer}/...)
    .withColumn("customer_name",
        split(col("path"), "/").getItem(6)  # adjust if needed
    )
    
    # Normalize fields
    .withColumn("file_name_lower", lower(col("file_name")))
    .withColumn("path_lower", lower(col("path")))
    
    # Proposal detection
    .withColumn("is_proposal_candidate",
        (
            col("file_name_lower").contains("proposal") |
            col("path_lower").contains("proposal")
        ) &
        col("extension").isin(VALID_EXTENSIONS)
    )
    
    # File ID (important for downstream joins)
    .withColumn("file_id", sha2(col("path"), 256))
)
display(df)

StatementMeta(, dbab9ab1-6a0f-4194-8b8b-8ed1717414c7, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7c96269e-d7cd-4717-80fd-e59ba480c976)

In [10]:
(
    df
    .write
    .mode("overwrite")  # switch to append + merge later
    .format("delta")
    .saveAsTable(OUTPUT_TABLE)
)

StatementMeta(, dbab9ab1-6a0f-4194-8b8b-8ed1717414c7, 12, Finished, Available, Finished, False)

## Transform Data

In [18]:
proposal_files = spark.table("proposal_file_inventory") \
    .filter("is_proposal_candidate = true AND extension = '.pptx'")

display(proposal_files)

StatementMeta(, dbab9ab1-6a0f-4194-8b8b-8ed1717414c7, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f340506d-4275-4e93-af00-4c9378c77666)

## Extract all text from slides and tables

In [15]:


def extract_pptx_content(file_path):
    prs = Presentation(file_path)
    slides_data = []

    for slide_index, slide in enumerate(prs.slides, start=1):
        slide_text = []
        tables = []

        for shape in slide.shapes:
            # Regular text boxes, titles, bullets
            if hasattr(shape, "text") and shape.text:
                slide_text.append(shape.text.strip())

            # Tables
            if getattr(shape, "has_table", False):
                table_rows = []

                for row in shape.table.rows:
                    cells = [cell.text.strip() for cell in row.cells]
                    table_rows.append(cells)

                tables.append(table_rows)

        # Flatten tables into searchable text
        table_text_blocks = []
        for table in tables:
            for row in table:
                table_text_blocks.append(" | ".join(row))

        combined_text = "\n".join(
            [*slide_text, *table_text_blocks]
        )

        slides_data.append({
            "slide_number": slide_index,
            "slide_text": "\n".join(slide_text),
            "tables": tables,
            "table_text": "\n".join(table_text_blocks),
            "combined_text": combined_text
        })

    return slides_data

StatementMeta(, dbab9ab1-6a0f-4194-8b8b-8ed1717414c7, 22, Finished, Available, Finished, False)

In [16]:

def process_file(row):
    try:
        slides = extract_pptx_content(row.path)
        
        output = []
        
        for slide in slides:
            output.append({
                "file_id": row.file_id,
                "customer_name": row.customer_name,
                "file_path": row.path,
                "slide_number": slide["slide_number"],
                "slide_text": slide["slide_text"],
                "table_text": slide["table_text"],
                "tables_json": json.dumps(slide["tables"]),
                "combined_text": slide["combined_text"]
            })
        
        return output
    
    except Exception as e:
        return [{
            "file_id": row.file_id,
            "customer_name": row.customer_name,
            "file_path": row.path,
            "slide_number": None,
            "slide_text": None,
            "table_text": None,
            "tables_json": None,
            "combined_text": None,
            "error": str(e)
        }]

StatementMeta(, dbab9ab1-6a0f-4194-8b8b-8ed1717414c7, 23, Finished, Available, Finished, False)

In [19]:
rdd = proposal_files.rdd.flatMap(process_file)

slides_df = spark.createDataFrame(rdd)

StatementMeta(, dbab9ab1-6a0f-4194-8b8b-8ed1717414c7, 26, Finished, Available, Finished, False)

In [20]:
display(slides_df)

StatementMeta(, dbab9ab1-6a0f-4194-8b8b-8ed1717414c7, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d57ea146-c321-4d18-a694-861aa98f48c7)

In [22]:
(
    slides_df
    .write
    .mode("overwrite")  # switch to append + merge later
    .format("delta")
    .saveAsTable(SLIDES_OUTPUT_TABLE)
)

StatementMeta(, dbab9ab1-6a0f-4194-8b8b-8ed1717414c7, 29, Finished, Available, Finished, False)